In [1]:
import sys

!{sys.executable} -m pip install \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
chromadb \
sentence-transformers \
pypdf \
langchain-huggingface


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ========================
# LangChain Imports
# ========================
from pathlib import Path

from langchain_community.document_loaders import (
    TextLoader
)

c:\Users\meytb\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ========================
# Documents Path
# ========================

# ========================
# Documents Path
# ========================

DOCS_PATH = Path(r"C:\Users\meytb\finshield-ai\docs")

print(
    f"Documents directory: {DOCS_PATH}"
)


Documents directory: C:\Users\meytb\finshield-ai\docs


In [4]:
# ========================
# Load Documents
# ========================

documents = []

for file_path in DOCS_PATH.glob("*.md"):
    
    loader = TextLoader(
        
        str(file_path),
        
        encoding="utf-8"
    )
    
    docs = loader.load()
    
    documents.extend(docs)

print(
    f"Loaded {len(documents)} documents."
)

Loaded 4 documents.


In [5]:
# ========================
# Document Preview
# ========================

print(
    documents[0].page_content[:2000]
)

# Credit Scoring System

## Overview

The credit scoring system is designed to estimate the probability that a borrower will default on a loan obligation. The model predicts the Probability of Default (PD) using machine learning techniques and engineered financial risk indicators.

## The scoring system supports:

loan approval decisions
borrower risk segmentation
portfolio risk management
explainable AI analysis
underwriting support

The system was developed using the Home Credit Default Risk dataset.

## Modeling Pipeline

The credit scoring pipeline includes:

Data preprocessing
Missing value handling
Feature engineering
Categorical encoding
Model training
Probability calibration
Threshold optimization
Risk segmentation
Explainability analysis

The final production model uses:

XGBoost classifier
CatBoostEncoder for categorical variables
probability calibration using sigmoid scaling
SHAP explainability
## Main Predictive Features

The model relies heavily on financial and behavioral

In [6]:
# ========================
# Document Metadata
# ========================

documents[0].metadata

{'source': 'C:\\Users\\meytb\\finshield-ai\\docs\\credit_doc.md'}

#  Document Chunking

This section splits the financial documents into smaller semantic chunks for retrieval-augmented generation (RAG).

Chunking improves:
- semantic search quality
- retrieval precision
- LLM contextual understanding

In [7]:
# ========================
# Markdown Splitter
# ========================

from langchain_text_splitters import (
    
    MarkdownHeaderTextSplitter,
    
    RecursiveCharacterTextSplitter
)
# ========================
# Markdown Headers
# ========================

headers_to_split_on = [

    ("#", "Header 1"),
    
    ("##", "Header 2"),
    
    ("###", "Header 3"),
]
# ========================
# Markdown Splitter
# ========================

markdown_splitter = (
    
    MarkdownHeaderTextSplitter(
        
        headers_to_split_on=
        headers_to_split_on
    )
)
# ========================
# Markdown Chunks
# ========================

md_chunks = []

for doc in documents:
    
    splits = markdown_splitter.split_text(
        
        doc.page_content
    )
    
    md_chunks.extend(splits)

print(
    f"Markdown chunks: "
    f"{len(md_chunks)}"
)

Markdown chunks: 35


In [12]:
# ========================
# Recursive Splitter
# ========================

from langchain_core.documents import Document

recursive_splitter = (
    
    RecursiveCharacterTextSplitter(
        
        chunk_size=1000,
        
        chunk_overlap=200,
        
        separators=["\n\n", "\n", " "]
    )
)
# ========================
# Smart Recursive Chunking
# ========================

chunk_size = 1000
min_chunk_size = 600
max_merged_size = 1400

final_chunks = []

for chunk in md_chunks:
    
    if len(chunk.page_content) <= chunk_size:
        
        final_chunks.append(chunk)
    
    else:
        
        split_chunks = recursive_splitter.split_documents(
            [chunk]
        )
        
        final_chunks.extend(
            split_chunks
        )

# Merge very small fragments with the previous chunk to avoid too many tiny chunks.
merged_chunks = []
for chunk in final_chunks:
    
    if (
        merged_chunks 
        and len(chunk.page_content) < min_chunk_size
        and len(merged_chunks[-1].page_content) + len(chunk.page_content) <= max_merged_size
    ):
        
        merged_chunks[-1] = Document(
            page_content=
            merged_chunks[-1].page_content
            + "\n\n"
            + chunk.page_content,
            metadata=merged_chunks[-1].metadata
        )
    
    else:
        
        merged_chunks.append(chunk)

final_chunks = merged_chunks

print(
    f"Final chunks: "
    f"{len(final_chunks)}"
)

Final chunks: 9


In [ ]:
# ========================
# Chunk Size Analysis
# ========================

import numpy as np

chunk_lengths = [

    len(chunk.page_content)
    
    for chunk in final_chunks
]

sizes = np.array(chunk_lengths)

print(
    f"Average Chunk Size: "
    f"{sizes.mean():.0f}"
)

print(
    f"Median Chunk Size: "
    f"{np.median(sizes):.0f}"
)

print(
    f"Max Chunk Size: "
    f"{sizes.max()}"
)

print(
    f"Min Chunk Size: "
    f"{sizes.min()}"
)

print()
for p in [10, 25, 50, 75, 90]:
    print(f"{p}th percentile: {np.percentile(sizes, p):.0f}")

print()
for lower, upper in [(0, 400), (400, 600), (600, 800), (800, 1000), (1000, 1200), (1200, 1600)]:
    count = ((sizes >= lower) & (sizes < upper)).sum()
    print(f"Chunks {lower}-{upper-1}: {count}")

Average Chunk Size: 1057
Median Chunk Size: 1169
Max Chunk Size: 1289
Min Chunk Size: 252

10th percentile: 811
25th percentile: 1054
50th percentile: 1169
75th percentile: 1237
90th percentile: 1273

Chunks 0-399: 1
Chunks 400-599: 0
Chunks 600-799: 0
Chunks 800-999: 1
Chunks 1000-1199: 4
Chunks 1200-1599: 3


In [16]:
# ========================
# Embedding Import
# ========================

from langchain_huggingface import (
    HuggingFaceEmbeddings
)
# ========================
# Embedding Model
# ========================

embedding_model = (
    
    HuggingFaceEmbeddings(
        
        model_name=
        "sentence-transformers/all-MiniLM-L6-v2"
    )
)

print(
    "Embedding model loaded."
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2585.78it/s]


Embedding model loaded.


# Create ChromaDB Vector Store

This section stores the financial knowledge base embeddings inside a ChromaDB vector database.

The vector store enables:
- semantic similarity search
- contextual retrieval
- retrieval-augmented generation

In [17]:

# ========================
# Chroma Import
# ========================

from langchain_community.vectorstores import (
    Chroma
)

In [18]:
# ========================
# Create ChromaDB
# ========================

vectorstore = Chroma.from_documents(
    
    documents=final_chunks,
    
    embedding=embedding_model,
    
    persist_directory="chroma_db"
)

print(
    "ChromaDB vector store created."
)

ChromaDB vector store created.


In [19]:
# ========================
# Verify ChromaDB
# ========================

print(
    f"Stored chunks: "
    f"{vectorstore._collection.count()}"
)

Stored chunks: 45


In [20]:
# ========================
# Semantic Retrieval Test
# ========================

query = (
    
    "Why do low EXT_SOURCE scores increase risk?"
)

results = (
    
    vectorstore.similarity_search(
        
        query,
        
        k=3
    )
)

for i, result in enumerate(results):
    
    print(f"\n--- Result {i+1} ---\n")
    
    print(
        result.page_content[:1000]
    )


--- Result 1 ---

The most important predictive variables are:  
EXT_SOURCE_1
EXT_SOURCE_2
EXT_SOURCE_3  
These external scores strongly influence borrower risk predictions.  
Low EXT_SOURCE values generally increase default probability.

--- Result 2 ---

The credit scoring system uses threshold optimization to balance:  
borrower acceptance
default detection
operational cost  
Lower thresholds:  
increase recall
increase false positives  
Higher thresholds:  
reduce false positives
miss more defaults  
The selected threshold balances business risk and customer acceptance.

--- Result 3 ---

The dataset is highly imbalanced.  
Approximate class distribution:  
non-default: ~92%
default: ~8%  
The modeling pipeline accounts for imbalance using:  
threshold optimization
recall-focused evaluation
calibrated probability analysis


# API Mistral

mubazt0eFkAT8SBZM6vgkeLkYkJFPfs9

In [27]:
import os

os.environ["MISTRAL_API_KEY"] = "mubazt0eFkAT8SBZM6vgkeLkYkJFPfs9"

In [28]:
# ========================
# Mistral Import
# ========================

from langchain_mistralai import (
    ChatMistralAI
)

In [ ]:
# ========================
# Mistral LLM
# ========================

llm = ChatMistralAI(
    
    model="open-mixtral-8x7b",
    
    temperature=0.2
)

print(
    "Mistral model connected."
)

Mistral model connected.


In [30]:
# ========================
# Retriever
# ========================

retriever = (
    
    vectorstore.as_retriever(
        
        search_kwargs={
            
            "k": 5
        }
    )
)

print(
    "Retriever created."
)

Retriever created.


In [31]:
# ========================
# Retrieval Test
# ========================

query = (
    
    "Why are low EXT_SOURCE scores risky?"
)

retrieved_docs = (
    
    retriever.invoke(query)
)

print(
    retrieved_docs[0].page_content
)

The most important predictive variables are:  
EXT_SOURCE_1
EXT_SOURCE_2
EXT_SOURCE_3  
These external scores strongly influence borrower risk predictions.  
Low EXT_SOURCE values generally increase default probability.


#  Build Financial RAG Chain

This section creates the full Retrieval-Augmented Generation (RAG) pipeline.

The chain combines:
- semantic retrieval
- contextual prompt augmentation
- Mistral AI generation

The assistant answers financial risk and fraud questions using the internal knowledge base.

In [35]:
# ========================
# LangChain Imports
# ========================

from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_core.runnables import (
    RunnablePassthrough
)

In [36]:
# ========================
# Financial RAG Prompt
# ========================

prompt_template = """
You are FinShield AI,
an expert financial risk assistant.

You answer questions about:
- credit scoring
- fraud detection
- underwriting
- explainable AI
- financial risk analysis

Use ONLY the provided context.

If the answer is not in the context,
say:
"I could not find the answer in the knowledge base."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = (
    
    ChatPromptTemplate.from_template(
        
        prompt_template
    )
)

print(
    "Prompt template created."
)

Prompt template created.


In [39]:
# ========================
# Format Retrieved Docs
# ========================

def format_docs(docs):
    
    return "\n\n".join(
        
        doc.page_content
        
        for doc in docs
    )

In [40]:
# ========================
# Modern RAG Chain
# ========================

rag_chain = (
    
    {
        "context":
        retriever | format_docs,
        
        "question":
        RunnablePassthrough()
    }
    
    | prompt
    
    | llm
    
    | StrOutputParser()
)

print(
    "Modern RAG chain created."
)

Modern RAG chain created.


In [41]:
# ========================
# First RAG Query
# ========================

question = (
    
    "Why do low EXT_SOURCE scores increase default risk?"
)

response = (
    
    rag_chain.invoke(
        
        question
    )
)

print(response)

Based on the provided context, low **EXT_SOURCE** scores (such as **EXT_SOURCE_1**, **EXT_SOURCE_2**, and **EXT_SOURCE_3**) increase default risk because these external scores are **strongly predictive of borrower risk**.

Specifically:
- The **EXT_SOURCE** variables are among the **most important predictive variables** in the credit scoring system.
- **Low EXT_SOURCE values generally correlate with a higher probability of default**, meaning borrowers with lower external scores are more likely to default on their loans.

The context does not provide further details on *why* these external scores are predictive (e.g., what they represent or how they are derived), but their strong influence on default risk is clearly established.
